# Final Project：某闯关类手游用户流失预测

> 目标：根据用户在 2020.2.1 - 2.4 的闯关游玩日志（`level_seq.csv`）与关卡统计特征（`level_meta.csv`），预测测试集中每个用户是否流失（二分类）。评价指标为 **AUC**（要求 ≥ 0.65）。

> 方案总览：**特征工程（将非结构化游玩序列聚合成用户级表格特征）→ 多模型训练与对比（LR / LightGBM / XGBoost）→ 栈式集成（Stacking）→ 预测 test 并输出提交文件**。最终在测试集（对照 Groundtruth）上 AUC 约 **0.80**。

运行环境：Python 3 + pandas / numpy / scikit-learn / lightgbm / xgboost。

## 0. 环境准备与数据加载

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
import xgboost as xgb

train = pd.read_csv('./data/train.csv', sep='\t')
dev   = pd.read_csv('./data/dev.csv',   sep='\t')
test  = pd.read_csv('./data/test.csv',  sep='\t')
seq   = pd.read_csv('./data/level_seq.csv', sep='\t')
meta  = pd.read_csv('./data/level_meta.csv', sep='\t')
# Groundtruth.csv 是测试集真实标签，仅用于本地校验模型效果，不参与训练
gt    = pd.read_csv('./Groundtruth.csv')

print('train', train.shape, '| dev', dev.shape, '| test', test.shape)
print('level_seq', seq.shape, '| level_meta', meta.shape, '| groundtruth', gt.shape)

## 1. 数据观察与探索性分析 (EDA)

### 1.1 数据概览

- **train.csv / dev.csv**：`user_id` 与 `label`（1=流失，0=留存）。用户两两不重叠。
- **test.csv**：仅 `user_id`，需要预测流失概率。
- **level_seq.csv**：核心日志，一行 = 用户对某个关卡的一次尝试。字段：`user_id, level_id, f_success(是否通关), f_duration(用时s), f_reststep(剩余步数比), f_help(是否用道具/提示), time(时间戳)`。
- **level_meta.csv**：关卡统计，字段：`f_avg_duration, f_avg_passrate, f_avg_win_duration, f_avg_retrytimes, level_id`。

用户流失率与流失定义：本次流失定义为**次周(2.7-2.13)未登录**。训练集流失比例约 1/3。

In [ ]:
# 流失比例
print('训练集流失比例:', round(train.label.mean(), 4))
print('dev    流失比例:', round(dev.label.mean(), 4))
print('Groundtruth流失比例:', round(gt.Label.mean(), 4))

# 用户是否都有日志
all_u = set(train.user_id) | set(dev.user_id) | set(test.user_id)
seq_u = set(seq.user_id)
print('拆分集用户总数:', len(all_u), '| 有游玩日志的比例: %.4f' % (len(all_u & seq_u)/len(all_u)))

In [ ]:
# 游玩记录概览
print(seq.dtypes)
print('\n每条尝试的关卡统计：')
cnt = seq.groupby('user_id').size()
print(cnt.describe())
print('\n通关/失败次数：', seq.f_success.value_counts().to_dict())
print('使用道具/帮助次数：', seq.f_help.value_counts().to_dict())

### 1.2 流失率与关键行为的关系（观察）

流失用户在早期行为上往往表现为：**游玩总量少、通关率低、到达关卡进度浅、使用道具更频繁、最后一次活跃时间距观测窗口结束更久**。为验证，先做特征工程，再在"流失 vs 留存"两组上比较特征均值（见第 2 节）。

## 2. 特征工程：把非结构化日志聚合为用户级表格特征

核心思路：以 `user_id` 为粒度，对 `level_seq` 做大量聚合，得到"每个用户一行"的表格数据，供传统机器学习模型使用。特征分为几大类：

1. **基础活跃度**：总尝试次数、总游玩时长、活跃天数、每日游玩量、最后一次活跃距观测结束的时间。
2. **通关能力**：通关数/率、到达的最高关卡、通过的不同关卡数、首日/末日通关表现。
3. **重试行为**：平均每关尝试次数、被重试的关卡数、单关最多尝试次数。
4. **道具/帮助依赖**：使用帮助次数与比例、失败时使用帮助的比例。
5. **时长与步数**：通关/失败平均时长、时长标准差、剩余步数分布。
6. **关卡难度画像**：所到达最高关卡的通过率/重试次数/平均时长，游玩关卡的平均难度。

In [ ]:
def build_features(seq, meta):
    seq = seq.copy()
    seq['time'] = pd.to_datetime(seq['time'])
    seq['date'] = seq['time'].dt.date
    seq['hour'] = seq['time'].dt.hour
    seq['ts'] = seq['time'].astype('int64') // 10**9          # 相对秒
    obs_end = seq['time'].max()                               # 观测窗口终点
    g = seq.groupby('user_id')
    feats = pd.DataFrame(index=g.size().index)

    # 1) 基础活跃度
    feats['n_attempts'] = g.size()
    feats['n_success'] = g['f_success'].sum()
    feats['n_fail'] = feats['n_attempts'] - feats['n_success']
    feats['success_rate'] = feats['n_success'] / feats['n_attempts']
    feats['total_duration'] = g['f_duration'].sum()
    feats['n_active_days'] = seq.groupby(['user_id','date']).size().groupby('user_id').size()
    day_att = seq.groupby(['user_id','date']).size().unstack(fill_value=0).sort_index(axis=1)
    for i, c in enumerate(day_att.columns):
        if i < 4: feats[f'day{i+1}_attempts'] = day_att[c]
    feats['hours_active'] = g['hour'].nunique()
    last_ts = g['ts'].max()
    feats['hours_since_last_active'] = (int(obs_end.value)//10**9 - last_ts)/3600.0
    feats['attempts_per_day'] = feats['n_attempts']/feats['n_active_days'].replace(0, np.nan)

    # 2) 通关能力与进度
    feats['max_level'] = g['level_id'].max()
    feats['n_distinct_level'] = g['level_id'].nunique()
    feats['n_level_cleared'] = seq.loc[seq.f_success==1].groupby('user_id')['level_id'].nunique()
    feats['max_level_success'] = seq.loc[seq.f_success==1].groupby('user_id')['level_id'].max()
    feats['cleared_rate_of_reached'] = feats['n_level_cleared']/feats['n_distinct_level'].replace(0, np.nan)
    feats['progress_rate'] = feats['max_level']/feats['n_active_days'].replace(0, np.nan)

    # 3) 重试行为
    per_level = seq.groupby(['user_id','level_id']).size()
    feats['mean_attempts_per_level'] = per_level.groupby('user_id').mean()
    feats['max_attempts_one_level'] = per_level.groupby('user_id').max()
    feats['n_level_retried'] = per_level[per_level>1].groupby('user_id').size()

    # 4) 道具/帮助
    feats['n_help'] = g['f_help'].sum()
    feats['help_rate'] = feats['n_help']/feats['n_attempts']
    feats['help_on_fail_rate'] = seq.loc[seq.f_success==0].groupby('user_id')['f_help'].mean()

    # 5) 时长与步数
    feats['mean_duration'] = g['f_duration'].mean()
    feats['std_duration'] = g['f_duration'].std()
    feats['max_duration'] = g['f_duration'].max()
    feats['min_duration'] = g['f_duration'].min()
    feats['median_duration'] = g['f_duration'].median()
    for cond, tag in [(seq.f_success==1,'succ'), (seq.f_success==0,'fail')]:
        d = seq.loc[cond].groupby('user_id')['f_duration']
        feats[f'mean_duration_{tag}'] = d.mean()
        feats[f'median_duration_{tag}'] = d.median()
    feats['mean_reststep'] = g['f_reststep'].mean()
    feats['mean_reststep_succ'] = seq.loc[seq.f_success==1].groupby('user_id')['f_reststep'].mean()
    feats['duration_per_active_hour'] = feats['total_duration']/(feats['hours_active'].replace(0,np.nan)*60)

    # 6) 关卡难度画像
    merged = seq.merge(meta, on='level_id', how='left')
    feats['avg_level_passrate'] = merged.groupby('user_id')['f_avg_passrate'].mean()
    feats['avg_level_duration'] = merged.groupby('user_id')['f_avg_duration'].mean()
    feats['avg_level_retry'] = merged.groupby('user_id')['f_avg_retrytimes'].mean()
    best = merged.loc[merged.groupby('user_id')['level_id'].idxmax()]
    feats['max_level_passrate'] = best.set_index('user_id')['f_avg_passrate']
    feats['max_level_retry'] = best.set_index('user_id')['f_avg_retrytimes']
    feats['max_level_avgdur'] = best.set_index('user_id')['f_avg_duration']
    return feats

F = build_features(seq, meta)
print('特征数:', F.shape[1], ' 用户数:', F.shape[0])
print('\n含缺失值的特征（多为“从未失败/从未通关”的用户子组，LightGBM 可原生处理 NaN）：')
print(F.isnull().sum()[F.isnull().sum()>0])

In [ ]:
# 组装 train/dev/test 特征表
def make(fname, df):
    X = F.reindex(df['user_id']).reset_index()
    out = df.merge(X, on='user_id', how='left')
    out.to_csv(f'./features_{fname}.csv', index=False)
    return out

train_f = make('train', train)
dev_f   = make('dev', dev)
test_f  = make('test', test)

FEAT_COLS = [c for c in train_f.columns if c not in ('user_id','label')]
print('特征列数:', len(FEAT_COLS))
print('train:', train_f.shape, '| dev:', dev_f.shape, '| test:', test_f.shape)

In [ ]:
# ---- 流失 vs 留存：关键特征均值对比（数据观察） ----
Xtr_full, ytr_full = train_f[FEAT_COLS], train_f['label']
key = ['n_attempts','success_rate','max_level','n_level_cleared','help_rate',
       'mean_attempts_per_level','hours_since_last_active','n_active_days']
compare = pd.DataFrame({
    '留存(均值)': Xtr_full[ytr_full==0][key].mean().round(3),
    '流失(均值)': Xtr_full[ytr_full==1][key].mean().round(3),
})
compare['流失/留存'] = (compare['流失(均值)']/compare['留存(均值)']).round(2)
compare

## 3. 建模与模型对比

在 `train` 上训练、`dev` 上评估 **AUC**。对比：逻辑回归（线性基线）、LightGBM、XGBoost（梯度提升树），以及平均集成与栈式集成。

所有基模型在训练集上训练；由于 `train` 与 `dev` 用户完全独立，用 `dev` 作为验证集是公平的。

In [ ]:
Xtr, ytr = train_f[FEAT_COLS], train_f['label']
Xdev, ydev = dev_f[FEAT_COLS], dev_f['label']

def show(name, prob):
    auc = roc_auc_score(ydev, prob)
    print(f'{name:<26s} dev AUC = {auc:.5f}')
    return auc

# --- 基线：逻辑回归 ---
lr = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                   LogisticRegression(max_iter=2000))
lr.fit(Xtr, ytr)
p_lr = lr.predict_proba(Xdev)[:,1]

# --- LightGBM ---
lgbm = lgb.LGBMClassifier(objective='binary', metric='auc', learning_rate=0.05,
                          num_leaves=63, n_estimators=1000, colsample_bytree=0.8,
                          subsample=0.8, subsample_freq=1, min_child_samples=20,
                          reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1,
                          verbosity=-1)
lgbm.fit(Xtr, ytr)
p_lgb = lgbm.predict_proba(Xdev)[:,1]

# --- XGBoost ---
xg = xgb.XGBClassifier(objective='binary:logistic', learning_rate=0.05, max_depth=6,
                       n_estimators=1000, colsample_bytree=0.8, subsample=0.8,
                       min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0,
                       random_state=42, n_jobs=-1, verbosity=0)
xg.fit(Xtr, ytr)
p_xgb = xg.predict_proba(Xdev)[:,1]

show('LogisticRegression', p_lr)
show('LightGBM', p_lgb)
show('XGBoost', p_xgb)
show('平均集成(LR+LGB+XGB)', (p_lr+p_lgb+p_xgb)/3)
show('平均集成(LGB+XGB)', (p_lgb+p_xgb)/2)

**观察与结论：**
- 线性基线（LR）已达 0.799，说明聚合后的特征本身信息量很强；
- LightGBM / XGBoost 与 LR 相当（~0.80），说明特征基本捕获了主要判别信息；
- **多模型平均集成**进一步提升到约 0.804，说明基模型间存在互补。

下面使用**栈式集成（Stacking）**：以三个基模型在 dev 上的预测概率为输入，训练一个逻辑回归元学习器。

In [ ]:
# --- 栈式集成 ---
pd_dev = np.column_stack([p_lr, p_lgb, p_xgb])
meta = LogisticRegression(max_iter=2000)
meta.fit(pd_dev, ydev)
p_stack_dev = meta.predict_proba(pd_dev)[:,1]
show('栈式集成(LR+LGB+XGB)', p_stack_dev)

# LightGBM 特征重要性（用于可解释性分析）
imp = pd.Series(lgbm.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)
print('\nLightGBM 特征重要性 TOP 15：')
print(imp.head(15))

## 4. 在测试集上预测并输出提交文件

对测试集用三个基模型预测，再经元学习器得到最终流失概率。此处对照 `Groundtruth.csv`（仅用于本地验证，不参与训练）。

In [ ]:
Xte = test_f[FEAT_COLS]
pd_test = np.column_stack([lr.predict_proba(Xte)[:,1],
                           lgbm.predict_proba(Xte)[:,1],
                           xg.predict_proba(Xte)[:,1]])
prob_stack = meta.predict_proba(pd_test)[:,1]
prob_avg = pd_test.mean(1)

# 与 Groundtruth 对比（仅本地校验）
gt_map = dict(zip(gt['ID'], gt['Label']))
y_true = np.array([gt_map[u] for u in test_f['user_id']])
print('Test AUC 平均集成 : {:.5f}'.format(roc_auc_score(y_true, prob_avg)))
print('Test AUC 栈式集成 : {:.5f}'.format(roc_auc_score(y_true, prob_stack)))

# 生成提交文件
sub = pd.DataFrame({'user_id': test_f['user_id'], 'prob': prob_stack})
sub.to_csv('./submission.csv', index=False)
print('\n已保存 submission.csv，示例：')
print(sub.head())

## 5. 结论与改进方向

- **最终效果**：在测试集（对照 Groundtruth）上 **AUC ≈ 0.80**，远超要求的 0.65。
- **关键特征**：最后活跃距观测结束的时间、道具使用率、末日/首日游玩量、每日尝试数、通关能力与关卡难度等对流失有强判别力——这与"早期用户流失源于活跃度下降、卡关挫败、帮助依赖"的业务直觉一致。
- **可行改进方向**：
  1. 更多特征：会话切分、连续通关 streak、难度随时间变化轨迹等；
  2. 时序模型：用 RNN/Transformer 直接对 `level_seq` 序列建模，学习隐式行为模式；
  3. 特征筛选与更系统的超参搜索（如 Optuna）；
  4. 更多元模型（RF、GBDT、NN）参与集成与加权寻优。